# 02 — Supervised Classification
**Owner: Jamie | Status: Waiting on 01**

## What this notebook does
- Loads train/val/test CSVs from data/processed/
- Builds TF-IDF vectors (5000 features): **fit on train** `cleaned_text`, **transform** val and test (same vocabulary)
- Trains two classifiers via OneVsRestClassifier:
  1. MultinomialNB — course method
  2. LogisticRegression class_weight='balanced' — new method
- Computes macro-F1, per-class F1, ROC-AUC, confusion matrix
- Saves trained models to data/models/

## Input files (already in repo — just pull)
- data/processed/train.csv
- data/processed/val.csv
- data/processed/test.csv

## Output files (committed to GitHub after this runs)
- data/models/tfidf.pkl
- data/models/nb_model.pkl
- data/models/lr_model.pkl

## How to run
Pull latest from GitHub. Run cells top to bottom.
DO NOT refit TF-IDF on val or test — train only.

In [10]:
# Import libraries
from pathlib import Path

import pandas as pd
import numpy as np
import re
import nltk
from better_profanity import profanity
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Download NLTK resources (run once)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Load splits from data/processed (run notebook from notebooks/ so ROOT resolves like 01_data_prep)
ROOT = Path("..")
PROCESSED_DIR = ROOT / "data" / "processed"

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

[nltk_data] Downloading package stopwords to /Users/kaffe/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/kaffe/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/kaffe/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [11]:
# ----------------------
# Text Preprocessing
# ----------------------
# Initialize tools
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
profanity.load_censor_words()


def preprocess_text(text):
    # 1. Convert to lowercase
    text = text.lower()
    # 2. Remove URLs (Reddit links)
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    # 3. Remove special characters, numbers, and extra spaces
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    # 4. Tokenize, remove stopwords, lemmatize, then drop profane tokens
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]

    # drop profane tokens
    # -------------DISSCUSSION-------------
    # maybe we shouldn't filter out profanity, because it represent some emotions like anger，disappointment
    # tokens = [t for t in tokens if not profanity.contains_profanity(t)]
    # -------------DISSCUSSION-------------

    # 5. Join tokens back to text
    return " ".join(tokens)



for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df["cleaned_text"] = df["text"].apply(preprocess_text)

print("Preprocessing done for train, val, test.")
print(f"  train: {len(train_df)} rows | val: {len(val_df)} rows | test: {len(test_df)} rows")

label_cols = [c for c in train_df.columns if c not in ("text", "cleaned_text")]
print("\nSample (train):")
print(train_df[["text", "cleaned_text"] + label_cols].head(3))

Preprocessing done for train, val, test.
  train: 43410 rows | val: 5426 rows | test: 5427 rows

Sample (train):
                                                text  \
0  My favourite food is anything I didn't have to...   
1  Now if he does off himself, everyone will thin...   
2                     WHY THE FUCK IS BAYLESS ISOING   

                                        cleaned_text  anger  disgust  fear  \
0                 favourite food anything didnt cook      0        0     0   
1  everyone think he laugh screwing people instea...      0        0     0   
2                                fuck bayless isoing      1        0     0   

   joy  sadness  surprise  neutral  
0    0        0         0        1  
1    0        0         0        1  
2    0        0         0        0  


In [12]:
# ----------------------
# TF-IDF Vectorization
# ----------------------
# Fit on train only; transform val/test with the same vocabulary (no leakage).
label_cols = [c for c in train_df.columns if c not in ("text", "cleaned_text")]

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
)

X_train = tfidf_vectorizer.fit_transform(train_df["cleaned_text"])
X_val = tfidf_vectorizer.transform(val_df["cleaned_text"])
X_test = tfidf_vectorizer.transform(test_df["cleaned_text"])

y_train = train_df[label_cols].to_numpy()
y_val = val_df[label_cols].to_numpy()
y_test = test_df[label_cols].to_numpy()

print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"X_train: {X_train.shape} (sparse) | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape} (sparse) | y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape} (sparse) | y_test:  {y_test.shape}")
# print(X_train)
# print(y_train)

Vocabulary size: 5000
X_train: (43410, 5000) (sparse) | y_train: (43410, 7)
X_val:   (5426, 5000) (sparse) | y_val:   (5426, 7)
X_test:  (5427, 5000) (sparse) | y_test:  (5427, 7)


In [14]:
# ----------------------
# Multinomial Naive Bayes (course baseline) + one validation hyperparameter sweep
# ----------------------

import joblib
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

# Directory for serialized artifacts (same layout as notebook header).
MODELS_DIR = ROOT / "data" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Hyperparameter we tune once on the validation set ---------------------------------
# MultinomialNB's `alpha` is additive (Laplace-style) smoothing for count / TF-IDF features.
# - Too small alpha → less smoothing → can overfit rare tokens.
# - Too large alpha → predictions shrink toward uniform → underfit.


alpha_candidates = [1e-5,1e-4,1e-3, 1e-2, 0.1, 1.0, 10.0]
best_alpha = None
best_val_macro_f1 = -1.0
val_scores_by_alpha = {}

for alpha in alpha_candidates:
    candidate_clf_tuned = OneVsRestClassifier(
        MultinomialNB(alpha=alpha),
        n_jobs=None,  # default serial; use n_jobs=-1 to parallelize the 7 binary estimators
    )
    candidate_clf_tuned.fit(X_train, y_train)
    y_val_pred = candidate_clf_tuned.predict(X_val)

    # Macro-F1: average F1 across emotion dimensions (rare labels weigh equally).
    macro_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)
    val_scores_by_alpha[alpha] = macro_f1
    if macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = macro_f1
        best_alpha = alpha

print("Validation macro-F1 by MultinomialNB alpha (higher is better on val):")
for a, s in sorted(val_scores_by_alpha.items()):
    marker = "  <-- selected" if a == best_alpha else ""
    print(f"  alpha={a:>6}  macro-F1={s:.4f}{marker}")
print(f"\nChosen alpha after one tuning round on X_val: {best_alpha}")

# --- Final model: refit on training data with validation-selected alpha -----------------
# We discard candidate fits and train one final `nb_clf` on X_train only using `best_alpha`.
nb_clf = OneVsRestClassifier(
    MultinomialNB(alpha=best_alpha),
)
nb_clf.fit(X_train, y_train)

# Sanity prints: training should be ≥ validation F1 if the model is not heavily regularized away.
y_train_pred = nb_clf.predict(X_train)
y_val_pred_final = nb_clf.predict(X_val)
train_macro_f1 = f1_score(y_train, y_train_pred, average="macro", zero_division=0)
val_macro_f1_final = f1_score(y_val, y_val_pred_final, average="macro", zero_division=0)
print(f"\nFinal NB — train macro-F1: {train_macro_f1:.4f} | val macro-F1: {val_macro_f1_final:.4f}")

# Persist artifacts for notebook 04 / deployment. Classifier expects TF-IDF from the same vectorizer.
joblib.dump(tfidf_vectorizer, MODELS_DIR / "tfidf.pkl")
joblib.dump(nb_clf, MODELS_DIR / "nb_model.pkl")
print(f"\nSaved TF-IDF vectorizer and NB model under: {MODELS_DIR.resolve()}")

Validation macro-F1 by MultinomialNB alpha (higher is better on val):
  alpha= 1e-05  macro-F1=0.2491
  alpha=0.0001  macro-F1=0.2491
  alpha= 0.001  macro-F1=0.2519  <-- selected
  alpha=  0.01  macro-F1=0.2519
  alpha=   0.1  macro-F1=0.2485
  alpha=   1.0  macro-F1=0.1900
  alpha=  10.0  macro-F1=0.0963

Chosen alpha after one tuning round on X_val: 0.001

Final NB — train macro-F1: 0.3502 | val macro-F1: 0.2519

Saved TF-IDF vectorizer and NB model under: /Users/kaffe/kaffe‘code/KTH_course/DM1590/EmotionRadar_1/data/models
